In [1]:
import cv2
import mediapipe as mp
import numpy as np

In [2]:
# Initialize MediaPipe Hands
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

In [3]:
# For webcam input:
video_path = 'D:\Ramana\Carrer\Projects\CV Gun Game\InputVid1.mp4'
cap = cv2.VideoCapture(video_path)

In [ ]:
with mp_hands.Hands(
    model_complexity=0,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
    max_num_hands=2) as hands:
    
    while cap.isOpened():
        success, image = cap.read()
        if not success:
            print("Ignoring empty camera frame.")
            continue
        
        # To improve performance, optionally mark the image as not writeable to pass by reference.
        image.flags.writeable = False
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = hands.process(image)
        
        # Draw the hand annotations on the image.
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        direction = "No hand detected"
        
        if results.multi_hand_landmarks:
            for hand_landmarks, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
                # Check if it's the right hand
                hand_label = handedness.classification[0].label
                
                # Draw all hand landmarks and connections
                mp_drawing.draw_landmarks(
                    image,
                    hand_landmarks,
                    mp_hands.HAND_CONNECTIONS,
                    mp_drawing_styles.get_default_hand_landmarks_style(),
                    mp_drawing_styles.get_default_hand_connections_style())
                
                # Only process right hand
                if hand_label == 'Right':
                    # Get thumb landmarks
                    thumb_tip = hand_landmarks.landmark[mp_hands.HandLandmark.THUMB_TIP]
                    thumb_mcp = hand_landmarks.landmark[mp_hands.HandLandmark.THUMB_MCP]
                    
                    # Calculate differences
                    vertical_diff = thumb_tip.y - thumb_mcp.y
                    horizontal_diff = thumb_tip.x - thumb_mcp.x
                    
                    # Determine direction
                    if abs(vertical_diff) > abs(horizontal_diff):
                        if vertical_diff < -0.05:  # Thumb up
                            direction = "Forward"
                        elif vertical_diff > 0.05:  # Thumb down
                            direction = "Backward"
                    else:
                        if horizontal_diff > 0.05:  # Thumb right
                            direction = "Right"
                        elif horizontal_diff < -0.05:  # Thumb left
                            direction = "Left"
        
        # Display direction
        cv2.putText(image, f"Direction: {direction}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        print(direction)
        
        # Flip the image horizontally for a selfie-view display
        cv2.imshow('MediaPipe Hands', cv2.flip(image, 1))
        
        if cv2.waitKey(5) & 0xFF == 27:
            break

In [5]:
cap.release()
cv2.destroyAllWindows()

In [6]:
import cv2
import mediapipe as mp

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

cap = cv2.VideoCapture(video_path)

with mp_hands.Hands(min_detection_confidence=0.5, min_tracking_confidence=0.5) as hands:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb_frame)

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                handedness = results.multi_handedness[0].classification[0].label
                if handedness == "Right":  # Track only the right hand
                    mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
                    
                    thumb_tip = hand_landmarks.landmark[mp_hands.HandLandmark.THUMB_TIP]
                    thumb_ip = hand_landmarks.landmark[mp_hands.HandLandmark.THUMB_IP]
                    index_mcp = hand_landmarks.landmark[mp_hands.HandLandmark.INDEX_FINGER_MCP]
                    
                    thumb_up = thumb_tip.y < thumb_ip.y and thumb_tip.y < index_mcp.y
                    thumb_down = thumb_tip.y > thumb_ip.y and thumb_tip.y > index_mcp.y
                    thumb_right = thumb_tip.x > thumb_ip.x
                    thumb_left = thumb_tip.x < thumb_ip.x
                    
                    if thumb_up:
                        print("Forward")
                        cv2.putText(frame, "Forward", (20, 40), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                    elif thumb_down:
                        print("Backward")                        
                        cv2.putText(frame, "Backward", (20, 40), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                    elif thumb_right:                        
                        cv2.putText(frame, "Right", (20, 40), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                        print("Right")
                    elif thumb_left:
                        print("Left")                        
                        cv2.putText(frame, "Left", (20, 40), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        
        cv2.imshow('Thumbs Up Tracker', frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


In [7]:
import cv2
import mediapipe as mp

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

cap = cv2.VideoCapture(video_path)

with mp_hands.Hands(min_detection_confidence=0.5, min_tracking_confidence=0.5) as hands:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb_frame)

        direction = None  # Initialize direction as None

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                handedness = results.multi_handedness[0].classification[0].label
                if handedness == "Left":  # Track only the right hand
                    mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
                    
                    thumb_tip = hand_landmarks.landmark[mp_hands.HandLandmark.THUMB_TIP]
                    thumb_ip = hand_landmarks.landmark[mp_hands.HandLandmark.THUMB_IP]
                    index_mcp = hand_landmarks.landmark[mp_hands.HandLandmark.INDEX_FINGER_MCP]
                    
                    thumb_up = thumb_tip.y < thumb_ip.y and thumb_tip.y < index_mcp.y
                    thumb_down = thumb_tip.y > thumb_ip.y and thumb_tip.y > index_mcp.y
                    thumb_right = thumb_tip.x > thumb_ip.x
                    thumb_left = thumb_tip.x < thumb_ip.x
                    
                    if thumb_up:
                        direction = "Forward"
                    elif thumb_down:
                        direction = "Backward"
                    elif thumb_right:
                        direction = "Right"
                    elif thumb_left:
                        direction = "Left"
        
        # Display direction on the frame
        if direction:
            # Put a background rectangle for better text visibility
            cv2.rectangle(frame, (10, 10), (200, 60), (0, 0, 0), -1)
            # Display the direction text
            cv2.putText(frame, f"Direction: {direction}", (20, 40), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        else:
            # Display "No direction detected" when no hand is found
            cv2.rectangle(frame, (10, 10), (300, 60), (0, 0, 0), -1)
            cv2.putText(frame, "No direction detected", (20, 40), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        
        cv2.imshow('Thumbs Up Tracker', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()